# Gen and mixed-prior comparison for 2D unfolding

Compare nominal generator-level dijet $\eta_{CM}$ with the mixed-Gaussian prior produced by `processForestSimple.C`. For every configured half-open $p_T^{ave}$ interval, this notebook writes raw-yield and independently normalized shape comparisons with `Mixed prior / Gen` in the lower panel.


<!-- detailed-workflow-guide -->

### Detailed workflow and prior interpretation

This notebook compares the nominal generator distribution with the deliberately modified prior used in unfolding studies. A prior is a starting probability shape for iterative Bayesian inversion, not an additional measured spectrum. If both histograms are normalized, their ratio tests shape only; otherwise it also contains their normalization difference.

For iteration $n$, Bayes unfolding updates truth probabilities using the response and the measured spectrum. More iterations reduce dependence on the initial prior but can amplify statistical fluctuations. The plotted prior distortion should therefore span plausible shape differences without introducing empty truth bins that the response cannot recover.

In [ ]:
# Cell role: initialize the reproducible Python/ROOT environment and shared helpers.
# Interpretation: No physics histogram is modified here; ROOT ownership is configured before files open.
# The preceding Markdown gives the equations and physics assumptions for this step.
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / 'CMakeLists.txt').is_file()
     and (candidate / 'hist_analysis').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Cannot locate jetAnalysis. Start Jupyter from its root.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root
ROOT = load_root(batch=True)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import (
    DIJET_DELTA_PHI_SELECTION_LABEL, DIJET_PTAVE_BINS,
    STANDARD_DIJET_ETA_CUT_INDEX,
)
from hist_analysis.python.histogram_io import (
    load_histogram, resolve_combined_file, resolve_direction_file,
)
from hist_analysis.python.histogram_ops import normalize_histogram
from hist_analysis.python.plotting import draw_closure
from hist_analysis.python.projections import project_semantic_th2

ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

from hist_analysis.config.histograms import TEST_DIJET_PTAVE_BINS
from hist_analysis.python.unfolding import as_pt_intervals, project_eta_by_pt


## Configuration

The default eta-cut index selects $|\eta_{CM}^{jet}|<1.9$. `PTAVE_BINS` uses the common unfolding intervals as $[low, high)$. Independent normalization isolates the shape change; raw plots retain weighted yields. Standard ROOT error propagation is used because binomial errors are not appropriate for a continuously reweighted prior.


In [ ]:
# Cell role: define and validate user-facing analysis configuration.
# Interpretation: Changing these values can change inputs, selections, binning, normalization, or outputs.
# The preceding Markdown gives the equations and physics assumptions for this step.
GENERATOR = 'embedding'       # embedding or pythia
DIRECTION = 'Pbgoing'        # pgoing, Pbgoing, or combined
FILE_STEM = 'jetId'
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.3, 2.4, 3.0)
ETA_CUT_INDEX = STANDARD_DIJET_ETA_CUT_INDEX
PTAVE_BIN_SET = 'standard'  # test or standard
PTAVE_BIN_SETS = {'test': TEST_DIJET_PTAVE_BINS, 'standard': DIJET_PTAVE_BINS}
PTAVE_BINS = as_pt_intervals(PTAVE_BIN_SETS[PTAVE_BIN_SET])
REBIN_ETA = 2
NORMALIZATION = 'integral'   # integral or bin_width
PLOT_RAW_DISTRIBUTIONS = False
RAW_RATIO_RANGE = (0.25, 2.5)
SHAPE_RATIO_RANGE = (0.25, 2.5)
SAVE_PNG = False
DRAW_GRID = True
OUTPUT_DIR = Path(os.environ.get(
    'UNFOLD2D_GEN_PRIOR_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'unfold2D_gen_and_prior',
))
GEN_HISTOGRAM_TEMPLATE = 'hGenDijetPtEtaCM_{eta_cut_index}'
PRIOR_HISTOGRAM_TEMPLATE = 'hGenDijetPtEtaCMMixedPrior_{eta_cut_index}'

if PTAVE_BIN_SET not in PTAVE_BIN_SETS:
    raise ValueError(f'Unsupported PTAVE_BIN_SET={PTAVE_BIN_SET!r}')
if GENERATOR not in {'embedding', 'pythia'}:
    raise ValueError(f'Unsupported GENERATOR={GENERATOR!r}')
if DIRECTION not in {'pgoing', 'Pbgoing', 'combined'}:
    raise ValueError(f'Unsupported DIRECTION={DIRECTION!r}')
if ETA_CUT_INDEX < 0 or ETA_CUT_INDEX >= len(ETA_CUTS):
    raise IndexError(f'Invalid ETA_CUT_INDEX={ETA_CUT_INDEX}')
if NORMALIZATION not in {'integral', 'bin_width'}:
    raise ValueError(f'Unsupported NORMALIZATION={NORMALIZATION!r}')
if not isinstance(PLOT_RAW_DISTRIBUTIONS, bool):
    raise TypeError('PLOT_RAW_DISTRIBUTIONS must be True or False')


## Resolve the MC file and load detached 2D histograms


In [ ]:
# Cell role: define and validate user-facing analysis configuration.
# Interpretation: Changing these values can change inputs, selections, binning, normalization, or outputs.
# The preceding Markdown gives the equations and physics assumptions for this step.
def mc_file(generator, direction):
    if direction == 'combined':
        return resolve_combined_file(BASE_DIR, generator, FILE_STEM)
    return resolve_direction_file(BASE_DIR, generator, direction, FILE_STEM)

DIRECTION_LABELS = {
    'pgoing': 'p-going', 'Pbgoing': 'Pb-going', 'combined': 'Combined',
}
INPUT_FILE = mc_file(GENERATOR, DIRECTION)
if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Missing configured ROOT file: {INPUT_FILE}')

gen_key = GEN_HISTOGRAM_TEMPLATE.format(eta_cut_index=ETA_CUT_INDEX)
prior_key = PRIOR_HISTOGRAM_TEMPLATE.format(eta_cut_index=ETA_CUT_INDEX)
gen_source = load_histogram(str(INPUT_FILE), gen_key)
prior_source = load_histogram(str(INPUT_FILE), prior_key)
if not gen_source.InheritsFrom('TH2') or not prior_source.InheritsFrom('TH2'):
    raise TypeError('Gen and mixed-prior inputs must both be TH2 histograms')
gen_source.Rebin2D(1, REBIN_ETA)
prior_source.Rebin2D(1, REBIN_ETA)
INPUT_FILE, gen_key, prior_key

gen_by_pt = project_eta_by_pt(gen_source, PTAVE_BINS, name_prefix='hGenEtaCM_priorComparison')
prior_by_pt = project_eta_by_pt(prior_source, PTAVE_BINS, name_prefix='hMixedPriorEtaCM_priorComparison')


## Compare Gen and mixed prior in each pTave interval

PDFs are written below `hist_analysis/output/unfold2D_gen_and_prior/` unless `UNFOLD2D_GEN_PRIOR_OUTPUT_DIR` is set. Results retain projections, ratios, and canvases for interactive inspection.


In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
comparison_results = {}
eta_cut = ETA_CUTS[ETA_CUT_INDEX]
eta_cut_tag = int(round(10.0 * eta_cut))
eta_x_range = (-eta_cut - 0.1, eta_cut + 0.1)

for ptave_index, ptave_range in enumerate(PTAVE_BINS):
    low, high = ptave_range
    if low >= high:
        raise ValueError(f'Invalid pTave interval: {ptave_range}')
    raw_shapes = {'Gen': gen_by_pt[ptave_index], 'Mixed prior': prior_by_pt[ptave_index]}
    normalized_shapes = {
        label: normalize_histogram(histogram, NORMALIZATION)
        for label, histogram in raw_shapes.items()
    }
    ptave_tag = f'{low:g}_{high:g}'.replace('.', 'p')
    base_tag = (f'{GENERATOR}_{DIRECTION}_gen_mixedPrior_'
                f'etaCM_{eta_cut_tag}_ptave_{ptave_tag}')
    annotations = (
        GENERATOR.capitalize(), DIRECTION_LABELS[DIRECTION],
        'Gen dijets, CM frame',
        f'{low:g} < p_{{T}}^{{ave}} < {high:g} GeV',
        f'|#eta_{{CM}}^{{jet}}| < {eta_cut:g}',
        'p_{T}^{Lead} > 50 GeV', 'p_{T}^{SubLead} > 40 GeV',
        DIJET_DELTA_PHI_SELECTION_LABEL,
    )
    raw_canvas = None
    raw_ratios = {}
    if PLOT_RAW_DISTRIBUTIONS:
        raw_canvas, raw_ratios = draw_closure(
            raw_shapes, 'Gen', title='', x_title='#eta_{CM}^{dijet}',
            y_title='dN/d#eta_{CM}^{dijet}', ratio_range=RAW_RATIO_RANGE,
            x_range=eta_x_range, annotations=annotations, grid=DRAW_GRID,
            headroom=1.6, draw_nominal_ratio=False, ratio_option='',
            style_indices={'Gen': 1, 'Mixed prior': 0},
            output=OUTPUT_DIR / f'{base_tag}_raw.pdf', save_png=SAVE_PNG,
            canvas_name=f'{base_tag}_raw',
        )
    normalized_canvas, normalized_ratios = draw_closure(
        normalized_shapes, 'Gen', title='', x_title='#eta_{CM}^{dijet}',
        y_title='1/N dN/d#eta_{CM}^{dijet}', ratio_range=SHAPE_RATIO_RANGE,
        x_range=eta_x_range, annotations=annotations, grid=DRAW_GRID,
        headroom=1.6, draw_nominal_ratio=False, ratio_option='',
        style_indices={'Gen': 1, 'Mixed prior': 0},
        output=OUTPUT_DIR / f'{base_tag}_normalized.pdf', save_png=SAVE_PNG,
        canvas_name=f'{base_tag}_normalized',
    )
    comparison_results[ptave_range] = {
        'raw_shapes': raw_shapes, 'raw_ratios': raw_ratios,
        'normalized_shapes': normalized_shapes,
        'normalized_ratios': normalized_ratios,
        'raw_canvas': raw_canvas, 'normalized_canvas': normalized_canvas,
    }
    print(ptave_range, {
        'gen_yield': raw_shapes['Gen'].Integral(),
        'prior_yield': raw_shapes['Mixed prior'].Integral(),
    })
    if raw_canvas is not None:
        display(raw_canvas)
    display(normalized_canvas)


## Numerical summary

Report raw yields and the non-empty normalized prior/Gen ratio range for each $p_T^{ave}$ interval.


In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
for ptave_range, result in comparison_results.items():
    ratio = result['normalized_ratios']['Mixed prior']
    values = [ratio.GetBinContent(i) for i in range(1, ratio.GetNbinsX() + 1)
              if ratio.GetBinContent(i) != 0.0]
    print(ptave_range, {
        'gen_raw_yield': result['raw_shapes']['Gen'].Integral(),
        'prior_raw_yield': result['raw_shapes']['Mixed prior'].Integral(),
        'gen_normalized_sum': result['normalized_shapes']['Gen'].Integral(),
        'prior_normalized_sum': result['normalized_shapes']['Mixed prior'].Integral(),
        'prior/gen range': (min(values), max(values)) if values else None,
    })
